# 08 · Validation Errors in Depth, JSON Schema & Custom Types

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 45 min

### 🎯 What you will learn
- The structure of `ValidationError`: `errors()`, `error_count()`, `json()`, `loc`, `type`, `msg`, `input`, `ctx`, `url`
- Turning errors into friendly API responses and translated messages
- Generating **JSON Schema** (`model_json_schema`): titles, descriptions, examples, `json_schema_extra`, modes
- Why JSON Schema matters (OpenAPI, **LLM structured output**, form generation)
- Custom types: `Annotated` recipes, and full control with `__get_pydantic_core_schema__`

In [1]:
import json
from typing import Annotated, Any, Literal
from pydantic import BaseModel, Field, ValidationError, ConfigDict, GetCoreSchemaHandler, TypeAdapter
from pydantic_core import core_schema

## 1. Anatomy of a `ValidationError`

In [2]:
class Address(BaseModel):
    city: str
    pincode: str = Field(pattern=r"^\d{6}$")

class Customer(BaseModel):
    name: str = Field(min_length=2)
    age: int = Field(ge=18)
    addresses: list[Address]

try:
    Customer.model_validate({"name": "A", "age": "ten", "addresses": [{"city": "Pune", "pincode": "4110"}, {}]})
except ValidationError as e:
    err = e
print(err.title, "-", err.error_count(), "errors")
for item in err.errors():
    print(item)

Customer - 5 errors
{'type': 'string_too_short', 'loc': ('name',), 'msg': 'String should have at least 2 characters', 'input': 'A', 'ctx': {'min_length': 2}, 'url': 'https://errors.pydantic.dev/2.14/v/string_too_short'}
{'type': 'int_parsing', 'loc': ('age',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'ten', 'url': 'https://errors.pydantic.dev/2.14/v/int_parsing'}
{'type': 'string_pattern_mismatch', 'loc': ('addresses', 0, 'pincode'), 'msg': "String should match pattern '^\\d{6}$'", 'input': '4110', 'ctx': {'pattern': '^\\d{6}$'}, 'url': 'https://errors.pydantic.dev/2.14/v/string_pattern_mismatch'}
{'type': 'missing', 'loc': ('addresses', 1, 'city'), 'msg': 'Field required', 'input': {}, 'url': 'https://errors.pydantic.dev/2.14/v/missing'}
{'type': 'missing', 'loc': ('addresses', 1, 'pincode'), 'msg': 'Field required', 'input': {}, 'url': 'https://errors.pydantic.dev/2.14/v/missing'}


| Key | Meaning |
|---|---|
| `loc` | path to the bad value: `("addresses", 0, "pincode")` |
| `type` | machine-readable error code: `string_too_short`, `int_parsing`, `missing`... |
| `msg` | human-readable message |
| `input` | the value that failed |
| `ctx` | extra details (`{"min_length": 2}`) |
| `url` | link to the documentation of that error type |

Useful options: `e.errors(include_url=False, include_input=False)` (don't echo user input/secrets back), `e.json(indent=2)`.

## 2. Friendly error responses

In [3]:
FRIENDLY = {
    "missing": "This field is required.",
    "string_too_short": "Please enter at least {min_length} characters.",
    "greater_than_equal": "Must be at least {ge}.",
    "int_parsing": "Please enter a whole number.",
    "string_pattern_mismatch": "Invalid format.",
}

def to_api_errors(e: ValidationError) -> dict[str, str]:
    out = {}
    for item in e.errors(include_url=False):
        field = ".".join(str(p) for p in item["loc"])
        template = FRIENDLY.get(item["type"], item["msg"])
        out[field] = template.format(**item.get("ctx", {}))
    return out

print(json.dumps(to_api_errors(err), indent=2))

{
  "name": "Please enter at least 2 characters.",
  "age": "Please enter a whole number.",
  "addresses.0.pincode": "Invalid format.",
  "addresses.1.city": "This field is required.",
  "addresses.1.pincode": "This field is required."
}


FastAPI does something similar automatically, returning HTTP **422** with this error list (lesson 10).

## 3. JSON Schema

Every model can describe itself as a **JSON Schema**, the standard format behind OpenAPI/Swagger docs, form generators, editors, and **structured output from LLMs**.

In [4]:
class Product(BaseModel):
    """A product in the catalogue."""
    model_config = ConfigDict(json_schema_extra={"examples": [{"sku": "PEN-1", "name": "Pen", "price": 9.5}]})

    sku: str = Field(pattern=r"^[A-Z]+-\d+$", description="Stock keeping unit")
    name: str = Field(min_length=1, max_length=80, title="Product name")
    price: float = Field(gt=0, description="Price in INR")
    category: Literal["office", "books", "toys"] = "office"
    tags: list[str] = Field(default_factory=list, max_length=5)

print(json.dumps(Product.model_json_schema(), indent=2))

{
  "description": "A product in the catalogue.",
  "examples": [
    {
      "name": "Pen",
      "price": 9.5,
      "sku": "PEN-1"
    }
  ],
  "properties": {
    "sku": {
      "description": "Stock keeping unit",
      "pattern": "^[A-Z]+-\\d+$",
      "title": "Sku",
      "type": "string"
    },
    "name": {
      "maxLength": 80,
      "minLength": 1,
      "title": "Product name",
      "type": "string"
    },
    "price": {
      "description": "Price in INR",
      "exclusiveMinimum": 0,
      "title": "Price",
      "type": "number"
    },
    "category": {
      "default": "office",
      "enum": [
        "office",
        "books",
        "toys"
      ],
      "title": "Category",
      "type": "string"
    },
    "tags": {
      "items": {
        "type": "string"
      },
      "maxItems": 5,
      "title": "Tags",
      "type": "array"
    }
  },
  "required": [
    "sku",
    "name",
    "price"
  ],
  "title": "Product",
  "type": "object"
}


In [5]:
class Order(BaseModel):
    id: int
    items: list[Product]

schema = Order.model_json_schema()
print(list(schema), "→ nested models live in $defs:", list(schema["$defs"]))
print(TypeAdapter(list[int]).json_schema())

['$defs', 'properties', 'required', 'title', 'type'] → nested models live in $defs: ['Product']
{'items': {'type': 'integer'}, 'type': 'array'}


### Validation vs serialization schemas

Input and output can differ (computed fields, serializers, aliases). Choose with `mode`:

In [6]:
from pydantic import computed_field

class Rect(BaseModel):
    w: float
    h: float
    @computed_field
    @property
    def area(self) -> float:
        return self.w * self.h

print(list(Rect.model_json_schema(mode="validation")["properties"]))
print(list(Rect.model_json_schema(mode="serialization")["properties"]))

['w', 'h']
['w', 'h', 'area']


### Use case: structured output from an LLM

Most LLM APIs accept a JSON Schema and return JSON matching it; Pydantic validates the reply:

```python
schema = Product.model_json_schema()        # send this as the response format / tool schema
reply_text = call_llm(prompt, json_schema=schema)        # pseudo-code
product = Product.model_validate_json(reply_text)        # never trust the model blindly: validate!
```
SDKs like Anthropic's and OpenAI's and libraries like `instructor` or `pydantic-ai` build on exactly this.

## 4. Custom types

### Recipe 1: `Annotated` with validators + serializers (most cases)

In [7]:
from pydantic import AfterValidator, PlainSerializer, BeforeValidator, WithJsonSchema

def parse_inr(v: Any) -> Any:
    if isinstance(v, str):
        v = v.replace("₹", "").replace(",", "").strip()
    return v

INR = Annotated[
    float,
    BeforeValidator(parse_inr),
    AfterValidator(lambda v: round(v, 2)),
    PlainSerializer(lambda v: f"₹{v:,.2f}", return_type=str, when_used="json"),
    WithJsonSchema({"type": "string", "pattern": r"^₹[\d,]+(\.\d{2})?$", "examples": ["₹1,250.00"]}),
]

class Invoice(BaseModel):
    total: INR

inv = Invoice(total="₹1,250.499")
print(inv.total, inv.model_dump_json(), Invoice.model_json_schema()["properties"]["total"])

1250.5 {"total":"₹1,250.50"} {'examples': ['₹1,250.00'], 'pattern': '^₹[\\d,]+(\\.\\d{2})?$', 'title': 'Total', 'type': 'string'}


### Recipe 2: a class with `__get_pydantic_core_schema__` (full control)

Use this for your own value classes:

In [8]:
class PhoneNumber(str):
    """A normalised Indian mobile number like '+919876543210'."""

    @classmethod
    def __get_pydantic_core_schema__(cls, source: Any, handler: GetCoreSchemaHandler) -> core_schema.CoreSchema:
        return core_schema.no_info_after_validator_function(
            cls.validate,
            core_schema.str_schema(strip_whitespace=True),            # first: must be a string
            serialization=core_schema.plain_serializer_function_ser_schema(str),
        )

    @classmethod
    def validate(cls, value: str) -> "PhoneNumber":
        digits = "".join(c for c in value if c.isdigit())
        if len(digits) == 10:
            digits = "91" + digits
        if not (len(digits) == 12 and digits.startswith("91") and digits[2] in "6789"):
            raise ValueError("not a valid Indian mobile number")
        return cls("+" + digits)

    @property
    def masked(self) -> str:
        return self[:3] + "******" + self[-4:]

class Contact(BaseModel):
    name: str
    phone: PhoneNumber

c = Contact(name="Asha", phone=" 98765-43210 ")
print(c.phone, type(c.phone).__name__, c.phone.masked, c.model_dump_json())
try:
    Contact(name="X", phone="12345")
except ValidationError as e:
    print("❌", e.errors()[0]["msg"])

+919876543210 PhoneNumber +91******3210 {"name":"Asha","phone":"+919876543210"}
❌ Value error, not a valid Indian mobile number


`core_schema` offers building blocks (`int_schema`, `str_schema`, `union_schema`, `chain_schema`, validator/serializer functions). Third-party integrations (e.g. `pydantic-extra-types`, bson `ObjectId`) use this hook.

## 📝 Summary
- `ValidationError.errors()` gives `loc`, `type`, `msg`, `input`, `ctx`; map `type` to friendly/translated messages; hide inputs when needed.
- `model_json_schema()` / `TypeAdapter.json_schema()` produce JSON Schema (with `$defs`, descriptions, examples) for OpenAPI, forms and LLM structured output.
- Custom types: `Annotated[...]` with validators/serializers/`WithJsonSchema`, or `__get_pydantic_core_schema__` for full control.

## 🏋️ Practice
1. Write a function that converts a `ValidationError` into `{"field": ["messages", ...]}` (multiple errors per field).
2. Generate the JSON Schema for a `Booking` model and save it to `booking.schema.json`.
3. Create a custom `Slug` class (lowercase, hyphens) using `__get_pydantic_core_schema__`.